# Background Augmentation — Focused Test Notebook

This notebook tests `AddRandomBackground` in isolation and as part of the pipeline.
It was created after discovering that the transform was silently broken in Albumentations v2
(the `always_apply` argument was being passed as `p`, setting `p=0.0` permanently).

**Sections:**
1. How dark pixel detection works — visualize the mask
2. Pixel brightness distribution — why threshold=30
3. The transform in isolation — many runs, confirm variety
4. After ShiftScaleRotate — the real production use case
5. Pixel-level verification — prove card pixels are preserved
6. Across multiple seed images — confirm it generalizes
7. Regression test — confirm the v2 bug is fixed and p= works correctly

In [ ]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
import sys

sys.path.append(os.path.abspath('..'))
from src.utils.augmentations import AddRandomBackground, get_spatial_color_transforms

RAW_DIR = '../data/raw'

# Pick a representative seed image to use throughout the notebook
SAMPLE = 'red_diamond_1_solid.jpg'

img_bgr = cv2.imread(os.path.join(RAW_DIR, SAMPLE))
image_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
image = cv2.resize(image_rgb, (224, 224))  # match pipeline size

print(f'Loaded: {SAMPLE}')
print(f'Shape: {image.shape}  dtype: {image.dtype}')

%matplotlib inline

---
## 1. How Dark Pixel Detection Works

`AddRandomBackground` identifies which pixels to replace using a simple brightness threshold:
1. Convert the image to grayscale
2. Mark every pixel with brightness **<= 30** as background (to be replaced)
3. Leave all brighter pixels unchanged (card content)

Threshold = 30 covers both the original dark rounded corners of the seed images
and the solid black padding (value = 0) introduced by `ShiftScaleRotate`.

In [ ]:
gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
_, mask = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY_INV)

# Overlay: highlight pixels that will be replaced in red
overlay = image.copy()
overlay[mask > 0] = [220, 30, 30]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(image)
axes[0].set_title('Original (224x224)')
axes[0].axis('off')

axes[1].imshow(gray, cmap='gray')
axes[1].set_title('Grayscale')
axes[1].axis('off')

axes[2].imshow(mask, cmap='hot')
axes[2].set_title('Dark pixel mask\n(white = will be replaced, threshold <= 30)')
axes[2].axis('off')

axes[3].imshow(overlay)
axes[3].set_title('Overlay\n(red = pixels that will be replaced)')
axes[3].axis('off')

fig.suptitle('Step 1: Identifying Background Pixels', fontsize=13)
plt.tight_layout()
plt.show()

dark_px  = np.sum(mask > 0)
total_px = mask.size
print(f'Pixels marked for replacement: {dark_px:,} / {total_px:,}  ({100 * dark_px / total_px:.1f}% of image)')

---
## 2. Pixel Brightness Distribution

The threshold of 30 is a deliberate choice. This section shows why:
- Card content (shapes, shading patterns) has brightness well above 30
- Dark corners and rotation padding cluster near 0

The histogram makes the separation visible. The right panel shows how many dark pixels
each of the 81 seed images has — confirming the transform has something to work with.

In [ ]:
all_paths = sorted(glob.glob(os.path.join(RAW_DIR, '*.jpg')))

# Compute dark pixel % for every seed image
dark_pcts = []
for p in all_paths:
    img = cv2.resize(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB), (224, 224))
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, m = cv2.threshold(g, 30, 255, cv2.THRESH_BINARY_INV)
    dark_pcts.append(100 * np.sum(m > 0) / m.size)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram of grayscale values for the sample image
axes[0].hist(gray.flatten(), bins=60, color='steelblue', edgecolor='none', alpha=0.85)
axes[0].axvline(x=30, color='red', linestyle='--', linewidth=2, label='Threshold = 30')
axes[0].set_title(f'Brightness distribution — {SAMPLE}', fontsize=11)
axes[0].set_xlabel('Grayscale brightness (0 = black, 255 = white)')
axes[0].set_ylabel('Pixel count')
axes[0].legend()

# Dark pixel % across all 81 seed images
bars = axes[1].bar(range(len(dark_pcts)), dark_pcts, color='steelblue', alpha=0.85)
axes[1].axhline(y=np.mean(dark_pcts), color='red', linestyle='--',
                label=f'Mean: {np.mean(dark_pcts):.1f}%')
axes[1].set_title('Dark pixel % across all 81 seed images', fontsize=11)
axes[1].set_xlabel('Image index')
axes[1].set_ylabel('% pixels with brightness <= 30')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Dark pixel % across 81 images:')
print(f'  Min:  {min(dark_pcts):.1f}%')
print(f'  Max:  {max(dark_pcts):.1f}%')
print(f'  Mean: {np.mean(dark_pcts):.1f}%')

---
## 3. The Transform in Isolation

Running `AddRandomBackground` alone on the raw seed image (no rotation yet).
Every run should produce a **different** background — if they all look the same,
the transform is not working.

Each image title shows how many dark pixels remain after replacement
(ideally close to 0% — all dark pixels replaced).

In [ ]:
transform = AddRandomBackground(p=1.0)
print(f'Transform p value: {transform.p}  (must be 1.0 — if 0.0 the v2 bug is back)')

NUM_RUNS = 16
fig, axes = plt.subplots(4, 4, figsize=(16, 16))

for ax in axes.flatten():
    result = transform(image=image)['image']
    g = cv2.cvtColor(result, cv2.COLOR_RGB2GRAY)
    _, m = cv2.threshold(g, 30, 255, cv2.THRESH_BINARY_INV)
    remaining = 100 * np.sum(m > 0) / m.size
    ax.imshow(result)
    ax.set_title(f'dark remaining: {remaining:.1f}%', fontsize=9)
    ax.axis('off')

fig.suptitle(f'AddRandomBackground alone — {NUM_RUNS} runs (should vary each time)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. After ShiftScaleRotate — The Real Production Use Case

In the actual pipeline, `ShiftScaleRotate` runs **first** and introduces solid black
padding (value = 0) around the rotated card. `AddRandomBackground` then replaces
both the original dark corners **and** this new rotation padding.

This is the exact sequence used during training. Dark remaining should be close
to 0% — any leftover dark pixels are card pixels that are genuinely dark (not background).

In [ ]:
shift_rotate = A.ShiftScaleRotate(
    shift_limit=0.05, scale_limit=0.1, rotate_limit=45,
    border_mode=cv2.BORDER_CONSTANT, value=0, p=1.0)
add_bg = AddRandomBackground(p=1.0)

NUM_RUNS = 16
fig, axes = plt.subplots(4, 4, figsize=(16, 16))

for ax in axes.flatten():
    rotated = shift_rotate(image=image)['image']
    result  = add_bg(image=rotated)['image']
    g = cv2.cvtColor(result, cv2.COLOR_RGB2GRAY)
    _, m = cv2.threshold(g, 30, 255, cv2.THRESH_BINARY_INV)
    remaining = 100 * np.sum(m > 0) / m.size
    ax.imshow(result)
    ax.set_title(f'dark remaining: {remaining:.1f}%', fontsize=9)
    ax.axis('off')

fig.suptitle('ShiftScaleRotate -> AddRandomBackground — production order', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Pixel-Level Verification

This section proves two things:
1. **Dark pixels are replaced** — the background areas change
2. **Card pixels are preserved exactly** — the bright card content is untouched

The diff map should show changes **only** in the dark/padding areas, and the
max pixel difference in card areas should be exactly 0.

In [ ]:
# Force rotation to ensure black padding exists
rotated = shift_rotate(image=image)['image']
result  = add_bg(image=rotated)['image']

# Compute the mask on the rotated image (before AddRandomBackground)
g_rotated = cv2.cvtColor(rotated, cv2.COLOR_RGB2GRAY)
_, rot_mask = cv2.threshold(g_rotated, 30, 255, cv2.THRESH_BINARY_INV)
card_mask = rot_mask == 0   # True = card pixels (bright, should be unchanged)
bg_mask   = rot_mask > 0    # True = dark pixels (should be replaced)

# Compute absolute pixel diff between rotated and result
diff = np.abs(rotated.astype(np.int32) - result.astype(np.int32)).sum(axis=2)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(rotated)
axes[0].set_title('After ShiftScaleRotate\n(black padding visible)')
axes[0].axis('off')

axes[1].imshow(result)
axes[1].set_title('After AddRandomBackground\n(padding replaced)')
axes[1].axis('off')

im = axes[2].imshow(diff, cmap='hot')
axes[2].set_title('Pixel diff map\n(should only differ where padding was)')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046)

fig.suptitle('Pixel-Level Verification: Only Dark Areas Modified', fontsize=13)
plt.tight_layout()
plt.show()

card_diff = diff[card_mask]
bg_diff   = diff[bg_mask]
print('Card pixel area (bright, should be UNCHANGED):')
print(f'  Max diff:  {card_diff.max()}  (must be 0)')
print(f'  Mean diff: {card_diff.mean():.4f}  (must be 0.0)')
print()
print('Background pixel area (dark, should be REPLACED):')
print(f'  Min diff:  {bg_diff.min():.1f}')
print(f'  Mean diff: {bg_diff.mean():.1f}  (should be >> 0)')
print(f'  Max diff:  {bg_diff.max():.1f}')

---
## 6. Across Multiple Seed Images

Confirms the transform generalizes across different card types — different colors,
shapes, and shading patterns. Each row shows the original and two augmented variants.

In [ ]:
# Sample a spread of cards: different colors and shapes
test_names = [
    'red_diamond_1_solid.jpg',
    'green_squiggle_2_striped.jpg',
    'purple_oval_3_empty.jpg',
    'red_oval_2_empty.jpg',
    'green_diamond_3_striped.jpg',
    'purple_squiggle_1_solid.jpg',
]

pipeline = A.Compose([shift_rotate, add_bg])

fig, axes = plt.subplots(len(test_names), 3, figsize=(12, 4 * len(test_names)))

for row, name in enumerate(test_names):
    path = os.path.join(RAW_DIR, name)
    img  = cv2.resize(cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB), (224, 224))
    v1   = pipeline(image=img)['image']
    v2   = pipeline(image=img)['image']

    axes[row, 0].imshow(img)
    axes[row, 0].set_title(name.replace('.jpg', ''), fontsize=8)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(v1)
    axes[row, 1].set_title('Variant 1', fontsize=8)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(v2)
    axes[row, 2].set_title('Variant 2', fontsize=8)
    axes[row, 2].axis('off')

fig.suptitle('AddRandomBackground Across Different Card Types', fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. Regression Test — The Albumentations v2 Bug

This section documents the bug that caused the black background and verifies it is fixed.

**The bug:** In Albumentations v2, `ImageOnlyTransform.__init__` dropped the `always_apply`
parameter. Our old code called `super().__init__(always_apply, p)`, which passed
`always_apply=False` as the `p` positional argument — silently setting `p=0.0`.
With `p=0.0` the transform never fires, regardless of what `p` value you pass at construction.

**The fix:** Change to `super().__init__(p=p)` — explicit keyword arg, no ambiguity.

In [ ]:
# Reproduce the broken behaviour to confirm we understand the bug
class BrokenBackground(A.ImageOnlyTransform):
    """Recreates the original broken __init__ to demonstrate the bug."""
    def __init__(self, p=1.0):
        always_apply = False
        super().__init__(always_apply, p)   # BUG: passes False as p
    def apply(self, img, **params):
        result = img.copy()
        result[:] = [200, 50, 200]          # fill magenta — obvious if it fires
        return result

broken = BrokenBackground(p=1.0)
fixed  = AddRandomBackground(p=1.0)

print('--- p= value check ---')
print(f'BrokenBackground(p=1.0).p = {broken.p}   <- should be 1.0 but is {broken.p} (False=0)')
print(f'AddRandomBackground(p=1.0).p = {fixed.p}  <- correctly 1.0')

# Test both on an all-black image — every pixel should be replaced if the transform fires
black_img = np.zeros((64, 64, 3), dtype=np.uint8)

broken_result = broken(image=black_img)['image']
fixed_result  = fixed(image=black_img)['image']

print()
print('--- Fire test on all-black image ---')
print(f'BrokenBackground changed image: {not np.array_equal(black_img, broken_result)}')
print(f'  (False = transform never fired despite p=1.0)')
print(f'AddRandomBackground changed image: {not np.array_equal(black_img, fixed_result)}')
print(f'  Mean pixel value after fix: {fixed_result.mean():.1f}  (should be >> 0)')

# Visual confirmation
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(black_img)
axes[0].set_title('Input: all-black image')
axes[0].axis('off')

for i in range(3):
    r = broken(image=black_img)['image']
    axes[1].imshow(r)
axes[1].set_title(f'BrokenBackground (p={broken.p})\nNever fires — stays black')
axes[1].axis('off')

axes[2].imshow(fixed_result)
axes[2].set_title(f'AddRandomBackground (p={fixed.p})\nFires correctly — background replaced')
axes[2].axis('off')

# Show 3 more fixed results to confirm variety
result2 = fixed(image=black_img)['image']
axes[3].imshow(result2)
axes[3].set_title('Another run — different background each time')
axes[3].axis('off')

fig.suptitle('Regression Test: Albumentations v2 Bug (p=0) vs Fixed (p=1.0)', fontsize=13)
plt.tight_layout()
plt.show()